In [1]:
import numpy as np
import biotite
import biotite.structure as struc
from biotite.structure.io import load_structure
# from ccd_utils.ccd import get_component_atom_array

# lle_arr = get_component_atom_array(
#     'LLE', keep_leaving_atoms = False, keep_hydrogens =  False
# )

# struc.dihedral_backbone(lle_arr)

In [2]:
# to replace get_sequences function in RAPiDock/utils/inference_utils.py
three_to_one = {
    "ALA": "A",
    "ARG": "R",
    "ASN": "N",
    "ASP": "D",
    "CYS": "C",
    "GLN": "Q",
    "GLU": "E",
    "GLY": "G",
    "HIS": "H",
    "ILE": "I",
    "LEU": "L",
    "LYS": "K",
    "MET": "M",
    "PHE": "F",
    "PRO": "P",
    "SER": "S",
    "THR": "T",
    "TRP": "W",
    "TYR": "Y",
    "VAL": "V",
}

def extract_chain_sequences(pdb_files, three_to_one_mapping=three_to_one):

    batch_seqs = []
    if isinstance(pdb_files, str):
        pdb_files = [pdb_files]

    for pdb_file in pdb_files:
        atom_array = load_structure(pdb_file)
        chain_starts = struc.get_chain_starts(atom_array)
        residue_starts = struc.get_residue_starts(atom_array)
    
        seq = ':'.join([
            ''.join([
                three_to_one_mapping.get(res, 'X') 
                for res in atom_array.res_name[
                    residue_starts[
                        (residue_starts >= chain_starts[i]) & 
                        (residue_starts < (chain_starts[i + 1] if i + 1 < len(chain_starts) else len(atom_array)))
                    ]
                ]
            ])
            for i in range(len(chain_starts))
        ])

        batch_seqs.append(seq)
    
    return batch_seqs


In [3]:
example_peptide = './8c5l_C/peptide.pdb'
eg_arr = load_structure(example_peptide)
eg_seq = extract_chain_sequences(example_peptide)[0]
phi,psi,omega = struc.dihedral_backbone(eg_arr)

In [4]:
phi,psi,eg_seq

(array([        nan,  1.1996874 , -1.1761978 , -1.2780241 , -1.1171261 ,
        -1.2073543 , -0.90572304, -0.9618805 , -1.1445125 , -1.1216671 ,
        -0.9585758 , -1.0649606 , -1.0772507 , -0.84950227, -1.1251029 ,
        -1.160077  , -0.9809799 ,  1.282019  ], dtype=float32),
 array([ 1.178941  ,  0.58368737, -0.9777552 , -0.34956476, -0.8035009 ,
        -0.661466  , -0.9653134 , -0.6383756 , -0.6659549 , -0.75977516,
        -0.75781834, -0.9095269 , -0.8260488 , -0.8408457 , -0.75221825,
        -0.80676454, -0.1728509 ,         nan], dtype=float32),
 'DYKFSTLLMMLKDMHDSK')